<a href="https://colab.research.google.com/github/jintubhuyan-2000/Spatial-Gradient-of-Highway-Induced-Land-Cover-Change/blob/main/SECTION_4_9_%E2%80%94_NDBI_DYNAMICS_AND_DEVELOPMENT_INTENSITY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# =============================================================================
# TEZPUR–NORTH LAKHIMPUR HIGHWAY CORRIDOR
# SECTION 4.9 — NDBI DYNAMICS AND DEVELOPMENT INTENSITY
# =============================================================================
#
# PURPOSE:
#   Analyze Sentinel-2 NDBI dynamics between 2016 and 2025 and generate
#   publication-ready statistics, tables and figures for Section 4.9.
#
# INPUT RASTERS:
#   TZPR_NLP_NDBI_2016.tif
#   TZPR_NLP_NDBI_2020.tif
#   TZPR_NLP_NDBI_2025.tif
#   TZPR_NLP_NDBI_Change_2016_2025.tif
#
# OUTPUT:
#   01_NDBI_Descriptive_Statistics_2016_2025.csv
#   02_NDBI_Intensity_Classes_2016_2020_2025.csv
#   03_NDBI_Change_Statistics_2016_2025.csv
#   04_Annualized_NDBI_Change.csv
#   05_NDBI_Manuscript_Summary_Table.csv
#   06_NDBI_Manuscript_Values.txt
#   07_NDBI_Raster_Information.csv
#   08_NDBI_Change_Class_Summary.csv
#
# FIGURES:
#   Figure 15 — NDBI spatial distribution:
#       2016, 2020, 2025
#
#   Figure 16 — NDBI change:
#       2016–2025
#
# REQUIREMENTS:
#   rasterio
#   numpy
#   pandas
#   matplotlib
#
# =============================================================================


# =============================================================================
# 1. IMPORT LIBRARIES
# =============================================================================

import os
import warnings

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# =============================================================================
# 2. USER SETTINGS
# =============================================================================

INPUT_DIR = "/content/drive/MyDrive/TZPR_NLP_Research"

OUTPUT_DIR = os.path.join(
    INPUT_DIR,
    "NDBI_Analysis_4_9"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# Input files
NDBI_FILES = {
    2016: os.path.join(INPUT_DIR, "TZPR_NLP_NDBI_2016.tif"),
    2020: os.path.join(INPUT_DIR, "TZPR_NLP_NDBI_2020.tif"),
    2025: os.path.join(INPUT_DIR, "TZPR_NLP_NDBI_2025.tif"),
}

CHANGE_FILE = os.path.join(
    INPUT_DIR,
    "TZPR_NLP_NDBI_Change_2016_2025.tif"
)


# =============================================================================
# 3. PARAMETERS
# =============================================================================

# Valid NDBI range
NDBI_MIN = -1.0
NDBI_MAX = 1.0

# NDBI intensity classes
#
# These classes are intended for descriptive interpretation.
#
# Very low     < -0.20
# Low          -0.20 to 0.00
# Moderate     0.00 to 0.20
# High         0.20 to 0.40
# Very high    >= 0.40

INTENSITY_CLASSES = [
    ("Very low", -np.inf, -0.20),
    ("Low", -0.20, 0.00),
    ("Moderate", 0.00, 0.20),
    ("High", 0.20, 0.40),
    ("Very high", 0.40, np.inf),
]


# NDBI change thresholds
#
# Decrease: ΔNDBI < -0.05
# Stable:   -0.05 <= ΔNDBI <= +0.05
# Increase: ΔNDBI > +0.05
#
# Strong decrease: ΔNDBI < -0.15
# Strong increase: ΔNDBI > +0.15

CHANGE_THRESHOLD = 0.05
STRONG_CHANGE_THRESHOLD = 0.15


# =============================================================================
# 4. FUNCTIONS
# =============================================================================

def read_raster(path):
    """
    Read raster and convert nodata values to NaN.
    """

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"\nRaster not found:\n{path}\n"
        )

    with rasterio.open(path) as src:

        arr = src.read(1).astype("float64")

        profile = src.profile.copy()

        nodata = src.nodata

        transform = src.transform

        crs = src.crs

        width = src.width

        height = src.height

        bounds = src.bounds

        dtype = src.dtypes[0]

        resolution_x = abs(transform.a)

        resolution_y = abs(transform.e)

    # Handle nodata
    if nodata is not None:
        arr[arr == nodata] = np.nan

    # Handle physically invalid values
    arr[(arr < NDBI_MIN) | (arr > NDBI_MAX)] = np.nan

    return {
        "array": arr,
        "profile": profile,
        "crs": crs,
        "transform": transform,
        "width": width,
        "height": height,
        "bounds": bounds,
        "dtype": dtype,
        "resolution_x": resolution_x,
        "resolution_y": resolution_y,
    }


# -----------------------------------------------------------------------------


def valid_values(arr):
    """
    Return finite raster values.
    """

    return arr[np.isfinite(arr)]


# -----------------------------------------------------------------------------


def calculate_area_ha(raster_info, valid_pixel_count):
    """
    Calculate area in hectares.

    For projected rasters:
        pixel area is calculated from raster resolution.

    For geographic rasters:
        approximate geographic pixel area is calculated using latitude.
    """

    transform = raster_info["transform"]
    crs = raster_info["crs"]

    # Projected CRS
    if crs is not None and crs.is_projected:

        pixel_area_m2 = abs(transform.a * transform.e)

        return (
            valid_pixel_count *
            pixel_area_m2 /
            10000.0
        )

    # Geographic CRS
    elif crs is not None and crs.is_geographic:

        arr_shape = (
            raster_info["height"],
            raster_info["width"]
        )

        rows = np.arange(arr_shape[0])

        cols = np.arange(arr_shape[1])

        # Use raster center latitude
        center_row = int(len(rows) / 2)

        _, center_lat = rasterio.transform.xy(
            transform,
            center_row,
            0,
            offset="center"
        )

        lat_rad = np.deg2rad(center_lat)

        dlon = abs(transform.a)

        dlat = abs(transform.e)

        earth_radius = 6378137.0

        pixel_width = (
            earth_radius *
            np.cos(lat_rad) *
            np.deg2rad(dlon)
        )

        pixel_height = (
            earth_radius *
            np.deg2rad(dlat)
        )

        pixel_area_m2 = pixel_width * pixel_height

        return (
            valid_pixel_count *
            pixel_area_m2 /
            10000.0
        )

    else:

        # Fallback
        pixel_area_m2 = abs(
            transform.a * transform.e
        )

        return (
            valid_pixel_count *
            pixel_area_m2 /
            10000.0
        )


# -----------------------------------------------------------------------------


def descriptive_statistics(year, raster_info):
    """
    Calculate descriptive NDBI statistics.
    """

    arr = raster_info["array"]

    values = valid_values(arr)

    if len(values) == 0:
        raise ValueError(
            f"No valid NDBI pixels found for {year}."
        )

    percentiles = np.percentile(
        values,
        [5, 25, 50, 75, 95]
    )

    area_ha = calculate_area_ha(
        raster_info,
        len(values)
    )

    result = {

        "Year": year,

        "Mean_NDBI":
            np.mean(values),

        "Median_NDBI":
            np.median(values),

        "Minimum":
            np.min(values),

        "Maximum":
            np.max(values),

        "Std":
            np.std(values),

        "P05":
            percentiles[0],

        "P25":
            percentiles[1],

        "P50":
            percentiles[2],

        "P75":
            percentiles[3],

        "P95":
            percentiles[4],

        "Area_ha":
            area_ha,

        "Area_km2":
            area_ha / 100.0,

        "Valid_Pixels":
            len(values),
    }

    return result


# -----------------------------------------------------------------------------


def calculate_intensity_classes(
    year,
    raster_info
):
    """
    Calculate percentage and area of each NDBI intensity class.
    """

    arr = raster_info["array"]

    values = valid_values(arr)

    total = len(values)

    total_area = calculate_area_ha(
        raster_info,
        total
    )

    results = []

    for class_name, lower, upper in INTENSITY_CLASSES:

        mask = (
            (values >= lower) &
            (values < upper)
        )

        count = np.sum(mask)

        percentage = (
            count / total * 100
        )

        area_ha = (
            total_area *
            percentage / 100
        )

        results.append({

            "Year": year,

            "NDBI_Class":
                class_name,

            "Lower_Bound":
                lower if np.isfinite(lower)
                else None,

            "Upper_Bound":
                upper if np.isfinite(upper)
                else None,

            "Pixels":
                count,

            "Area_ha":
                area_ha,

            "Area_km2":
                area_ha / 100.0,

            "Percent":
                percentage,
        })

    return results


# -----------------------------------------------------------------------------


def change_statistics(
    change_array,
    raster_info
):
    """
    Calculate NDBI change statistics.
    """

    values = valid_values(change_array)

    if len(values) == 0:
        raise ValueError(
            "No valid NDBI change pixels found."
        )

    total_pixels = len(values)

    total_area = calculate_area_ha(
        raster_info,
        total_pixels
    )

    mean_change = np.mean(values)

    median_change = np.median(values)

    minimum = np.min(values)

    maximum = np.max(values)

    std = np.std(values)

    increase_mask = (
        values > CHANGE_THRESHOLD
    )

    decrease_mask = (
        values < -CHANGE_THRESHOLD
    )

    stable_mask = (
        (values >= -CHANGE_THRESHOLD) &
        (values <= CHANGE_THRESHOLD)
    )

    strong_increase_mask = (
        values > STRONG_CHANGE_THRESHOLD
    )

    strong_decrease_mask = (
        values < -STRONG_CHANGE_THRESHOLD
    )

    increase_pixels = np.sum(
        increase_mask
    )

    decrease_pixels = np.sum(
        decrease_mask
    )

    stable_pixels = np.sum(
        stable_mask
    )

    strong_increase_pixels = np.sum(
        strong_increase_mask
    )

    strong_decrease_pixels = np.sum(
        strong_decrease_mask
    )

    increase_percent = (
        increase_pixels /
        total_pixels * 100
    )

    decrease_percent = (
        decrease_pixels /
        total_pixels * 100
    )

    stable_percent = (
        stable_pixels /
        total_pixels * 100
    )

    strong_increase_percent = (
        strong_increase_pixels /
        total_pixels * 100
    )

    strong_decrease_percent = (
        strong_decrease_pixels /
        total_pixels * 100
    )

    return {

        "Mean_Change":
            mean_change,

        "Median_Change":
            median_change,

        "Minimum_Change":
            minimum,

        "Maximum_Change":
            maximum,

        "Std_Change":
            std,

        "Increase_Area_ha":
            total_area *
            increase_percent / 100,

        "Increase_Area_km2":
            total_area *
            increase_percent / 100 /
            100,

        "Increase_Percent":
            increase_percent,

        "Decrease_Area_ha":
            total_area *
            decrease_percent / 100,

        "Decrease_Area_km2":
            total_area *
            decrease_percent / 100 /
            100,

        "Decrease_Percent":
            decrease_percent,

        "Stable_Area_ha":
            total_area *
            stable_percent / 100,

        "Stable_Area_km2":
            total_area *
            stable_percent / 100 /
            100,

        "Stable_Percent":
            stable_percent,

        "Strong_Increase_Area_ha":
            total_area *
            strong_increase_percent / 100,

        "Strong_Increase_Area_km2":
            total_area *
            strong_increase_percent / 100 /
            100,

        "Strong_Increase_Percent":
            strong_increase_percent,

        "Strong_Decrease_Area_ha":
            total_area *
            strong_decrease_percent / 100,

        "Strong_Decrease_Area_km2":
            total_area *
            strong_decrease_percent / 100 /
            100,

        "Strong_Decrease_Percent":
            strong_decrease_percent,

        "Total_Area_ha":
            total_area,

        "Total_Area_km2":
            total_area / 100,

        "Valid_Pixels":
            total_pixels,
    }


# -----------------------------------------------------------------------------


def calculate_change_classes(
    change_array,
    raster_info
):
    """
    Create detailed NDBI change classes.
    """

    values = valid_values(change_array)

    total = len(values)

    total_area = calculate_area_ha(
        raster_info,
        total
    )

    classes = [

        (
            "Strong decrease",
            -np.inf,
            -STRONG_CHANGE_THRESHOLD
        ),

        (
            "Moderate decrease",
            -STRONG_CHANGE_THRESHOLD,
            -CHANGE_THRESHOLD
        ),

        (
            "Stable",
            -CHANGE_THRESHOLD,
            CHANGE_THRESHOLD
        ),

        (
            "Moderate increase",
            CHANGE_THRESHOLD,
            STRONG_CHANGE_THRESHOLD
        ),

        (
            "Strong increase",
            STRONG_CHANGE_THRESHOLD,
            np.inf
        ),
    ]

    results = []

    for name, lower, upper in classes:

        mask = (
            (values >= lower) &
            (values < upper)
        )

        count = np.sum(mask)

        percentage = (
            count /
            total *
            100
        )

        area_ha = (
            total_area *
            percentage /
            100
        )

        results.append({

            "Change_Class":
                name,

            "Pixels":
                count,

            "Area_ha":
                area_ha,

            "Area_km2":
                area_ha / 100,

            "Percent":
                percentage,

        })

    return results


# -----------------------------------------------------------------------------


def raster_information(
    year,
    raster_info,
    filename
):
    """
    Extract raster metadata.
    """

    return {

        "Year":
            year,

        "Filename":
            filename,

        "Width":
            raster_info["width"],

        "Height":
            raster_info["height"],

        "CRS":
            str(raster_info["crs"]),

        "Resolution_X":
            raster_info["resolution_x"],

        "Resolution_Y":
            raster_info["resolution_y"],

        "Data_Type":
            raster_info["dtype"],

        "Valid_Pixels":
            np.sum(
                np.isfinite(
                    raster_info["array"]
                )
            ),
    }


# -----------------------------------------------------------------------------


def plot_nDBI_map(
    raster_info,
    title,
    output_file,
    vmin=-0.5,
    vmax=0.5
):
    """
    Plot an individual NDBI map.
    """

    arr = raster_info["array"]

    plt.figure(
        figsize=(10, 8)
    )

    im = plt.imshow(
        arr,
        cmap="RdYlGn_r",
        vmin=vmin,
        vmax=vmax
    )

    plt.title(
        title,
        fontsize=14,
        fontweight="bold"
    )

    plt.xlabel(
        "Column"
    )

    plt.ylabel(
        "Row"
    )

    cbar = plt.colorbar(
        im,
        fraction=0.046,
        pad=0.04
    )

    cbar.set_label(
        "NDBI",
        fontsize=11
    )

    plt.tight_layout()

    plt.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# -----------------------------------------------------------------------------


def plot_change_map(
    raster_info,
    output_file
):
    """
    Plot NDBI change map.
    """

    arr = raster_info["array"]

    plt.figure(
        figsize=(10, 8)
    )

    max_abs = np.nanpercentile(
        np.abs(arr),
        98
    )

    max_abs = max(
        max_abs,
        STRONG_CHANGE_THRESHOLD
    )

    im = plt.imshow(
        arr,
        cmap="RdBu_r",
        vmin=-max_abs,
        vmax=max_abs
    )

    plt.title(
        "NDBI Change, 2016–2025",
        fontsize=14,
        fontweight="bold"
    )

    plt.xlabel(
        "Column"
    )

    plt.ylabel(
        "Row"
    )

    cbar = plt.colorbar(
        im,
        fraction=0.046,
        pad=0.04
    )

    cbar.set_label(
        "ΔNDBI (2025 − 2016)",
        fontsize=11
    )

    plt.tight_layout()

    plt.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# -----------------------------------------------------------------------------


def plot_figure_15(
    rasters
):
    """
    Create publication-ready Figure 15:
    NDBI maps for 2016, 2020 and 2025.
    """

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(17, 6)
    )

    years = [2016, 2020, 2025]

    for ax, year in zip(
        axes,
        years
    ):

        arr = rasters[year]["array"]

        im = ax.imshow(
            arr,
            cmap="RdYlGn_r",
            vmin=-0.5,
            vmax=0.5
        )

        ax.set_title(
            str(year),
            fontsize=13,
            fontweight="bold"
        )

        ax.set_xlabel(
            "Column"
        )

        ax.set_ylabel(
            "Row"
        )

    cbar = fig.colorbar(
        im,
        ax=axes,
        fraction=0.025,
        pad=0.02
    )

    cbar.set_label(
        "NDBI",
        fontsize=11
    )

    fig.suptitle(
        "Figure 15. Spatial distribution of Sentinel-2 NDBI across the highway corridor",
        fontsize=14,
        fontweight="bold"
    )

    plt.tight_layout()

    output_file = os.path.join(
        OUTPUT_DIR,
        "Figure_15_NDBI_2016_2020_2025.png"
    )

    plt.savefig(
        output_file,
        dpi=400,
        bbox_inches="tight"
    )

    plt.close()

    print(
        f"Figure 15 saved:\n{output_file}"
    )


# -----------------------------------------------------------------------------


def plot_figure_16(
    change_info
):
    """
    Create publication-ready Figure 16:
    NDBI change 2016–2025.
    """

    arr = change_info["array"]

    plt.figure(
        figsize=(10, 8)
    )

    max_abs = np.nanpercentile(
        np.abs(arr),
        98
    )

    max_abs = max(
        max_abs,
        STRONG_CHANGE_THRESHOLD
    )

    im = plt.imshow(
        arr,
        cmap="RdBu_r",
        vmin=-max_abs,
        vmax=max_abs
    )

    cbar = plt.colorbar(
        im,
        fraction=0.046,
        pad=0.04
    )

    cbar.set_label(
        "ΔNDBI (2025 − 2016)",
        fontsize=12
    )

    plt.title(
        "Figure 16. Spatial distribution of NDBI change between 2016 and 2025",
        fontsize=14,
        fontweight="bold"
    )

    plt.xlabel(
        "Column"
    )

    plt.ylabel(
        "Row"
    )

    plt.tight_layout()

    output_file = os.path.join(
        OUTPUT_DIR,
        "Figure_16_NDBI_Change_2016_2025.png"
    )

    plt.savefig(
        output_file,
        dpi=400,
        bbox_inches="tight"
    )

    plt.close()

    print(
        f"Figure 16 saved:\n{output_file}"
    )


# =============================================================================
# 5. LOAD DATA
# =============================================================================

print("\n")
print("=" * 80)
print("SECTION 4.9 — NDBI DYNAMICS AND DEVELOPMENT INTENSITY")
print("=" * 80)

print("\nLoading NDBI rasters...")

rasters = {}

for year, path in NDBI_FILES.items():

    print(
        f"  Loading {year}: "
        f"{os.path.basename(path)}"
    )

    rasters[year] = read_raster(path)

change_info = read_raster(
    CHANGE_FILE
)

change_array = change_info["array"]


# =============================================================================
# 6. DESCRIPTIVE STATISTICS
# =============================================================================

print("\nCalculating descriptive statistics...")

descriptive_results = []

for year in [2016, 2020, 2025]:

    result = descriptive_statistics(
        year,
        rasters[year]
    )

    descriptive_results.append(
        result
    )

descriptive_df = pd.DataFrame(
    descriptive_results
)

descriptive_file = os.path.join(
    OUTPUT_DIR,
    "01_NDBI_Descriptive_Statistics_2016_2025.csv"
)

descriptive_df.to_csv(
    descriptive_file,
    index=False
)

print(
    f"Saved: {descriptive_file}"
)


# =============================================================================
# 7. NDBI INTENSITY CLASSES
# =============================================================================

print("\nCalculating NDBI intensity classes...")

intensity_results = []

for year in [2016, 2020, 2025]:

    year_results = calculate_intensity_classes(
        year,
        rasters[year]
    )

    intensity_results.extend(
        year_results
    )

intensity_df = pd.DataFrame(
    intensity_results
)

intensity_file = os.path.join(
    OUTPUT_DIR,
    "02_NDBI_Intensity_Classes_2016_2020_2025.csv"
)

intensity_df.to_csv(
    intensity_file,
    index=False
)

print(
    f"Saved: {intensity_file}"
)


# =============================================================================
# 8. NDBI CHANGE STATISTICS
# =============================================================================

print("\nCalculating NDBI change statistics...")

change_stats = change_statistics(
    change_array,
    change_info
)

change_stats_df = pd.DataFrame(
    [change_stats]
)

change_stats_file = os.path.join(
    OUTPUT_DIR,
    "03_NDBI_Change_Statistics_2016_2025.csv"
)

change_stats_df.to_csv(
    change_stats_file,
    index=False
)

print(
    f"Saved: {change_stats_file}"
)


# =============================================================================
# 9. ANNUALIZED NDBI CHANGE
# =============================================================================

print("\nCalculating annualized NDBI change...")

years_between = 2025 - 2016

mean_change = (
    change_stats["Mean_Change"]
)

annualized_change = (
    mean_change /
    years_between
)

annualized_df = pd.DataFrame({

    "Start_Year": [2016],

    "End_Year": [2025],

    "Years": [years_between],

    "Mean_NDBI_2016": [
        descriptive_df.loc[
            descriptive_df["Year"] == 2016,
            "Mean_NDBI"
        ].iloc[0]
    ],

    "Mean_NDBI_2025": [
        descriptive_df.loc[
            descriptive_df["Year"] == 2025,
            "Mean_NDBI"
        ].iloc[0]
    ],

    "Net_Mean_NDBI_Change": [
        mean_change
    ],

    "Annualized_Mean_NDBI_Change": [
        annualized_change
    ],

    "Gross_Increase_Area_ha": [
        change_stats[
            "Increase_Area_ha"
        ]
    ],

    "Gross_Decrease_Area_ha": [
        change_stats[
            "Decrease_Area_ha"
        ]
    ],

    "Stable_Area_ha": [
        change_stats[
            "Stable_Area_ha"
        ]
    ],
})

annualized_file = os.path.join(
    OUTPUT_DIR,
    "04_Annualized_NDBI_Change.csv"
)

annualized_df.to_csv(
    annualized_file,
    index=False
)

print(
    f"Saved: {annualized_file}"
)


# =============================================================================
# 10. MANUSCRIPT SUMMARY TABLE
# =============================================================================

print("\nCreating manuscript summary table...")

stats_lookup = (
    descriptive_df
    .set_index("Year")
)

mean_2016 = stats_lookup.loc[
    2016,
    "Mean_NDBI"
]

mean_2020 = stats_lookup.loc[
    2020,
    "Mean_NDBI"
]

mean_2025 = stats_lookup.loc[
    2025,
    "Mean_NDBI"
]

median_2016 = stats_lookup.loc[
    2016,
    "Median_NDBI"
]

median_2020 = stats_lookup.loc[
    2020,
    "Median_NDBI"
]

median_2025 = stats_lookup.loc[
    2025,
    "Median_NDBI"
]

minimum_2016 = stats_lookup.loc[
    2016,
    "Minimum"
]

minimum_2020 = stats_lookup.loc[
    2020,
    "Minimum"
]

minimum_2025 = stats_lookup.loc[
    2025,
    "Minimum"
]

maximum_2016 = stats_lookup.loc[
    2016,
    "Maximum"
]

maximum_2020 = stats_lookup.loc[
    2020,
    "Maximum"
]

maximum_2025 = stats_lookup.loc[
    2025,
    "Maximum"
]

p25_2016 = stats_lookup.loc[
    2016,
    "P25"
]

p25_2020 = stats_lookup.loc[
    2020,
    "P25"
]

p25_2025 = stats_lookup.loc[
    2025,
    "P25"
]

p75_2016 = stats_lookup.loc[
    2016,
    "P75"
]

p75_2020 = stats_lookup.loc[
    2020,
    "P75"
]

p75_2025 = stats_lookup.loc[
    2025,
    "P75"
]

p95_2016 = stats_lookup.loc[
    2016,
    "P95"
]

p95_2020 = stats_lookup.loc[
    2020,
    "P95"
]

p95_2025 = stats_lookup.loc[
    2025,
    "P95"
]


summary_rows = [

    {
        "Indicator":
            "Mean NDBI",

        "2016":
            mean_2016,

        "2020":
            mean_2020,

        "2025":
            mean_2025,

        "2016_2025_Change":
            mean_2025 -
            mean_2016,
    },

    {
        "Indicator":
            "Median NDBI",

        "2016":
            median_2016,

        "2020":
            median_2020,

        "2025":
            median_2025,

        "2016_2025_Change":
            median_2025 -
            median_2016,
    },

    {
        "Indicator":
            "Minimum",

        "2016":
            minimum_2016,

        "2020":
            minimum_2020,

        "2025":
            minimum_2025,

        "2016_2025_Change":
            minimum_2025 -
            minimum_2016,
    },

    {
        "Indicator":
            "Maximum",

        "2016":
            maximum_2016,

        "2020":
            maximum_2020,

        "2025":
            maximum_2025,

        "2016_2025_Change":
            maximum_2025 -
            maximum_2016,
    },

    {
        "Indicator":
            "25th percentile",

        "2016":
            p25_2016,

        "2020":
            p25_2020,

        "2025":
            p25_2025,

        "2016_2025_Change":
            p25_2025 -
            p25_2016,
    },

    {
        "Indicator":
            "75th percentile",

        "2016":
            p75_2016,

        "2020":
            p75_2020,

        "2025":
            p75_2025,

        "2016_2025_Change":
            p75_2025 -
            p75_2016,
    },

    {
        "Indicator":
            "95th percentile",

        "2016":
            p95_2016,

        "2020":
            p95_2020,

        "2025":
            p95_2025,

        "2016_2025_Change":
            p95_2025 -
            p95_2016,
    },

    {
        "Indicator":
            "NDBI increase area (ha)",

        "2016":
            np.nan,

        "2020":
            np.nan,

        "2025":
            np.nan,

        "2016_2025_Change":
            change_stats[
                "Increase_Area_ha"
            ],
    },

    {
        "Indicator":
            "NDBI increase (%)",

        "2016":
            np.nan,

        "2020":
            np.nan,

        "2025":
            np.nan,

        "2016_2025_Change":
            change_stats[
                "Increase_Percent"
            ],
    },

    {
        "Indicator":
            "NDBI decrease area (ha)",

        "2016":
            np.nan,

        "2020":
            np.nan,

        "2025":
            np.nan,

        "2016_2025_Change":
            change_stats[
                "Decrease_Area_ha"
            ],
    },

    {
        "Indicator":
            "NDBI decrease (%)",

        "2016":
            np.nan,

        "2020":
            np.nan,

        "2025":
            np.nan,

        "2016_2025_Change":
            change_stats[
                "Decrease_Percent"
            ],
    },

    {
        "Indicator":
            "Stable NDBI area (ha)",

        "2016":
            np.nan,

        "2020":
            np.nan,

        "2025":
            np.nan,

        "2016_2025_Change":
            change_stats[
                "Stable_Area_ha"
            ],
    },

    {
        "Indicator":
            "Stable NDBI (%)",

        "2016":
            np.nan,

        "2020":
            np.nan,

        "2025":
            np.nan,

        "2016_2025_Change":
            change_stats[
                "Stable_Percent"
            ],
    },

    {
        "Indicator":
            "Strong increase area (ha)",

        "2016":
            np.nan,

        "2020":
            np.nan,

        "2025":
            np.nan,

        "2016_2025_Change":
            change_stats[
                "Strong_Increase_Area_ha"
            ],
    },

    {
        "Indicator":
            "Strong increase (%)",

        "2016":
            np.nan,

        "2020":
            np.nan,

        "2025":
            np.nan,

        "2016_2025_Change":
            change_stats[
                "Strong_Increase_Percent"
            ],
    },

    {
        "Indicator":
            "Strong decrease area (ha)",

        "2016":
            np.nan,

        "2020":
            np.nan,

        "2025":
            np.nan,

        "2016_2025_Change":
            change_stats[
                "Strong_Decrease_Area_ha"
            ],
    },

    {
        "Indicator":
            "Strong decrease (%)",

        "2016":
            np.nan,

        "2020":
            np.nan,

        "2025":
            np.nan,

        "2016_2025_Change":
            change_stats[
                "Strong_Decrease_Percent"
            ],
    },
]


summary_df = pd.DataFrame(
    summary_rows
)

summary_file = os.path.join(
    OUTPUT_DIR,
    "05_NDBI_Manuscript_Summary_Table.csv"
)

summary_df.to_csv(
    summary_file,
    index=False
)

print(
    f"Saved: {summary_file}"
)


# =============================================================================
# 11. MANUSCRIPT VALUES TXT
# =============================================================================

print("\nWriting manuscript values...")

txt_file = os.path.join(
    OUTPUT_DIR,
    "06_NDBI_Manuscript_Values.txt"
)

with open(
    txt_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "SECTION 4.9 — NDBI DYNAMICS AND DEVELOPMENT INTENSITY\n"
    )

    f.write(
        "=" * 70 + "\n\n"
    )

    f.write(
        "DESCRIPTIVE STATISTICS\n"
    )

    f.write(
        f"Mean NDBI 2016: "
        f"{mean_2016:.6f}\n"
    )

    f.write(
        f"Mean NDBI 2020: "
        f"{mean_2020:.6f}\n"
    )

    f.write(
        f"Mean NDBI 2025: "
        f"{mean_2025:.6f}\n"
    )

    f.write(
        f"Net mean NDBI change: "
        f"{mean_2025 - mean_2016:+.6f}\n"
    )

    f.write(
        f"Median NDBI 2016: "
        f"{median_2016:.6f}\n"
    )

    f.write(
        f"Median NDBI 2020: "
        f"{median_2020:.6f}\n"
    )

    f.write(
        f"Median NDBI 2025: "
        f"{median_2025:.6f}\n"
    )

    f.write(
        f"Net median NDBI change: "
        f"{median_2025 - median_2016:+.6f}\n"
    )

    f.write(
        f"Minimum NDBI 2016: "
        f"{minimum_2016:.6f}\n"
    )

    f.write(
        f"Minimum NDBI 2020: "
        f"{minimum_2020:.6f}\n"
    )

    f.write(
        f"Minimum NDBI 2025: "
        f"{minimum_2025:.6f}\n"
    )

    f.write(
        f"Maximum NDBI 2016: "
        f"{maximum_2016:.6f}\n"
    )

    f.write(
        f"Maximum NDBI 2020: "
        f"{maximum_2020:.6f}\n"
    )

    f.write(
        f"Maximum NDBI 2025: "
        f"{maximum_2025:.6f}\n"
    )

    f.write(
        "\nNDBI CHANGE 2016–2025\n"
    )

    f.write(
        f"Mean change: "
        f"{change_stats['Mean_Change']:+.6f}\n"
    )

    f.write(
        f"Median change: "
        f"{change_stats['Median_Change']:+.6f}\n"
    )

    f.write(
        f"Increase area: "
        f"{change_stats['Increase_Area_ha']:,.2f} ha\n"
    )

    f.write(
        f"Increase area: "
        f"{change_stats['Increase_Area_km2']:,.2f} km²\n"
    )

    f.write(
        f"Increase percentage: "
        f"{change_stats['Increase_Percent']:.2f}%\n"
    )

    f.write(
        f"Decrease area: "
        f"{change_stats['Decrease_Area_ha']:,.2f} ha\n"
    )

    f.write(
        f"Decrease area: "
        f"{change_stats['Decrease_Area_km2']:,.2f} km²\n"
    )

    f.write(
        f"Decrease percentage: "
        f"{change_stats['Decrease_Percent']:.2f}%\n"
    )

    f.write(
        f"Stable area: "
        f"{change_stats['Stable_Area_ha']:,.2f} ha\n"
    )

    f.write(
        f"Stable area: "
        f"{change_stats['Stable_Area_km2']:,.2f} km²\n"
    )

    f.write(
        f"Stable percentage: "
        f"{change_stats['Stable_Percent']:.2f}%\n"
    )

    f.write(
        f"Strong increase area: "
        f"{change_stats['Strong_Increase_Area_ha']:,.2f} ha\n"
    )

    f.write(
        f"Strong increase percentage: "
        f"{change_stats['Strong_Increase_Percent']:.2f}%\n"
    )

    f.write(
        f"Strong decrease area: "
        f"{change_stats['Strong_Decrease_Area_ha']:,.2f} ha\n"
    )

    f.write(
        f"Strong decrease percentage: "
        f"{change_stats['Strong_Decrease_Percent']:.2f}%\n"
    )

    f.write(
        f"\nAnnualized mean NDBI change: "
        f"{annualized_change:+.6f} units/year\n"
    )

    f.write(
        f"Analysis period: "
        f"{years_between} years\n"
    )

    f.write(
        "\nTHRESHOLDS\n"
    )

    f.write(
        "NDBI increase: ΔNDBI > +0.05\n"
    )

    f.write(
        "NDBI decrease: ΔNDBI < -0.05\n"
    )

    f.write(
        "Stable: -0.05 <= ΔNDBI <= +0.05\n"
    )

    f.write(
        "Strong increase: ΔNDBI > +0.15\n"
    )

    f.write(
        "Strong decrease: ΔNDBI < -0.15\n"
    )

print(
    f"Saved: {txt_file}"
)


# =============================================================================
# 12. RASTER INFORMATION
# =============================================================================

print("\nExtracting raster information...")

raster_info_rows = []

for year in [2016, 2020, 2025]:

    raster_info_rows.append(
        raster_information(
            year,
            rasters[year],
            os.path.basename(
                NDBI_FILES[year]
            )
        )
    )

raster_info_rows.append(
    raster_information(
        "2016_2025_CHANGE",
        change_info,
        os.path.basename(
            CHANGE_FILE
        )
    )
)

raster_info_df = pd.DataFrame(
    raster_info_rows
)

raster_info_file = os.path.join(
    OUTPUT_DIR,
    "07_NDBI_Raster_Information.csv"
)

raster_info_df.to_csv(
    raster_info_file,
    index=False
)

print(
    f"Saved: {raster_info_file}"
)


# =============================================================================
# 13. DETAILED CHANGE CLASSES
# =============================================================================

print("\nCalculating detailed NDBI change classes...")

change_class_results = calculate_change_classes(
    change_array,
    change_info
)

change_class_df = pd.DataFrame(
    change_class_results
)

change_class_file = os.path.join(
    OUTPUT_DIR,
    "08_NDBI_Change_Class_Summary.csv"
)

change_class_df.to_csv(
    change_class_file,
    index=False
)

print(
    f"Saved: {change_class_file}"
)


# =============================================================================
# 14. FIGURE 15
# =============================================================================

print("\nCreating Figure 15...")

plot_figure_15(
    rasters
)


# =============================================================================
# 15. FIGURE 16
# =============================================================================

print("\nCreating Figure 16...")

plot_figure_16(
    change_info
)


# =============================================================================
# 16. OPTIONAL INDIVIDUAL MAPS
# =============================================================================

print("\nCreating individual NDBI maps...")

for year in [2016, 2020, 2025]:

    output_file = os.path.join(
        OUTPUT_DIR,
        f"NDBI_{year}.png"
    )

    plot_nDBI_map(
        rasters[year],
        f"Sentinel-2 NDBI — {year}",
        output_file
    )


# =============================================================================
# 17. PRINT KEY RESULTS
# =============================================================================

print("\n")
print("=" * 80)
print("KEY NDBI RESULTS")
print("=" * 80)

print(
    f"\nMean NDBI:"
)

print(
    f"  2016 = {mean_2016:.4f}"
)

print(
    f"  2020 = {mean_2020:.4f}"
)

print(
    f"  2025 = {mean_2025:.4f}"
)

print(
    f"\nNet mean change = "
    f"{mean_2025 - mean_2016:+.4f}"
)

print(
    f"\nMedian NDBI:"
)

print(
    f"  2016 = {median_2016:.4f}"
)

print(
    f"  2020 = {median_2020:.4f}"
)

print(
    f"  2025 = {median_2025:.4f}"
)

print(
    f"\nNDBI increase:"
)

print(
    f"  {change_stats['Increase_Area_ha']:,.2f} ha "
    f"({change_stats['Increase_Percent']:.2f}%)"
)

print(
    f"\nNDBI decrease:"
)

print(
    f"  {change_stats['Decrease_Area_ha']:,.2f} ha "
    f"({change_stats['Decrease_Percent']:.2f}%)"
)

print(
    f"\nStable NDBI:"
)

print(
    f"  {change_stats['Stable_Area_ha']:,.2f} ha "
    f"({change_stats['Stable_Percent']:.2f}%)"
)

print(
    f"\nStrong NDBI increase:"
)

print(
    f"  {change_stats['Strong_Increase_Area_ha']:,.2f} ha "
    f"({change_stats['Strong_Increase_Percent']:.2f}%)"
)

print(
    f"\nStrong NDBI decrease:"
)

print(
    f"  {change_stats['Strong_Decrease_Area_ha']:,.2f} ha "
    f"({change_stats['Strong_Decrease_Percent']:.2f}%)"
)

print(
    f"\nAnnualized mean NDBI change:"
)

print(
    f"  {annualized_change:+.6f} units/year"
)


# =============================================================================
# 18. OUTPUT DIRECTORY
# =============================================================================

print("\n")
print("=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

print(
    f"\nOutput directory:\n{OUTPUT_DIR}\n"
)

print(
    "\nGenerated files:"
)

for filename in sorted(
    os.listdir(OUTPUT_DIR)
):

    print(
        f"  ✓ {filename}"
    )

print(
    "\nFigures:"
)

print(
    "  ✓ Figure_15_NDBI_2016_2020_2025.png"
)

print(
    "  ✓ Figure_16_NDBI_Change_2016_2025.png"
)

print(
    "\nSection 4.9 analysis completed successfully."
)

# =============================================================================
# END OF SCRIPT
# =============================================================================
